# Part 4: Spatial Analysis, IV Identification, Counterfactuals

This notebook brings the spatial dimension and instrumental-variables identification to the analysis. We map the treatment, outcomes, and residuals; estimate 2SLS with distance to Wittenberg (Becker–Woessmann 2009) and an alternative bishop's-seat instrument; combine both for a Wooldridge over-identification test; correct standard errors for spatial autocorrelation via Conley (1999) HAC; and produce IV-implied counterfactual fertility paths.

**Headline new finding.** The IV story is a story of two outcomes:
- **Marriage rate** passes the Wooldridge over-identification test ($p=0.12$). Both instruments give consistent estimates. The result is now multiply identified.
- **CBR** fails the over-identification test ($p<0.001$). Different instruments give different IV coefficients — a strong signal of LATE heterogeneity or instrument-validity concerns.

## 1. Setup

In [ ]:
import sys
from pathlib import Path

# Add project root to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA_RAW = project_root / "data" / "raw" / "galloway_data"
DATA_PROCESSED = project_root / "data" / "processed"
OUTPUTS = project_root / "outputs" / "figures"
OUTPUTS.mkdir(exist_ok=True, parents=True)
TABLES = project_root / "outputs" / "tables"
TABLES.mkdir(exist_ok=True, parents=True)

print("Setup complete.")
print(f"Figures -> {OUTPUTS}")
print(f"Tables  -> {TABLES}")

DATA_RAW_GIS = project_root / "data" / "raw" / "gis_data"

from src.visualization.maps import (
    load_prussia_shapefile, map_catholic_share, map_fertility_change,
    map_polish_german_provinces, map_kulturkampf_residuals,
)
from src.visualization.plots import plot_counterfactual_paths
from src.analysis.regressions import run_iv_did, run_iv_did_multi
from src.analysis.conley_se import spatial_did_se
from src.data.centroids import load_centroids

panel = pd.read_parquet(DATA_PROCESSED / "analysis_panel.parquet")
gdf = load_prussia_shapefile(DATA_RAW_GIS / "German_Empire_1871_v.1.0.shp")
centroids = load_centroids()
panel_with_bishop = panel.merge(centroids[["Code", "km_bishop"]], on="Code", how="left")

print(f"Panel: {len(panel):,} obs, {panel['Code'].nunique()} counties.")
print(f"Shapefile: {len(gdf)} polygons.")
print(f"Centroids matched: {len(centroids)} counties; "
      f"with km_bishop: {centroids['km_bishop'].notna().sum()}.")

## 2. Map the treatment: Catholic share

The geography of religious composition in 1871 Prussia. The Catholic Rhineland (west), the Polish provinces (east), and the largely Protestant Prussian core in between.

In [ ]:
fig, ax = map_catholic_share(gdf, panel, savepath=str(OUTPUTS / "map1_catholic_share.png"))
plt.show()

**Interpretation.** Two large, geographically separated Catholic blocs surround a Protestant core: the western Rhineland counties (Cologne, Trier, Aachen) and the eastern Polish provinces (Posen, Bromberg, parts of West Prussia and Silesia). This spatial bimodality makes the average treatment effect particularly sensitive to *which* Catholic cluster drives identification — hence the importance of the Polish/German split in notebook 03.

## 3. Map the outcome: change in CBR pre vs post

In [ ]:
fig, ax = map_fertility_change(
    gdf, panel, pre_years=(1868, 1872), post_years=(1878, 1882),
    savepath=str(OUTPUTS / "map2_fertility_change.png"),
)
plt.show()

**Interpretation.** The map shows where CBR moved most between the late-1860s and the early-1880s. There's no clean Catholic-vs-Protestant pattern — fertility changes correlate with regions much more than with religion. This is what the regressions in notebook 02 also show: the unconditional Catholic–Protestant contrast is dominated by region-level dynamics.

## 4. Map the heterogeneity: Polish vs German Catholic provinces

In [ ]:
fig, ax = map_polish_german_provinces(gdf, panel, savepath=str(OUTPUTS / "map3_polish_german.png"))
plt.show()

**Interpretation.** Visualises the geography behind the central heterogeneity finding from notebook 03: the Catholic effect on fertility is concentrated in the eastern Polish provinces (Posen, Bromberg). The map makes vivid that "Catholic Prussia" was two very different societies in 1871.

## 5. Map the residuals from the baseline DiD

If the baseline TWFE residuals show strong spatial clustering, that's a sign of omitted regional confounders — and an argument for either Conley spatial HAC SEs (this notebook) or richer regional fixed effects (Year x Rb, in notebook 02).

In [ ]:
fig, ax = map_kulturkampf_residuals(
    gdf, panel, pre_years=(1868, 1872), post_years=(1873, 1878),
    savepath=str(OUTPUTS / "map4_residuals.png"),
)
plt.show()

**Interpretation.** Visible regional clustering of residuals (especially in the south-west and east) signals that there are spatial structures the baseline TWFE doesn't fully absorb. Two responses, both implemented later: (a) tighter regional FE (Year x Rb in notebook 02 absorbed most of this), and (b) Conley HAC SEs (section 8 below).

## 6. IV with distance to Wittenberg (Becker–Woessmann 2009)

Becker & Woessmann's QJE paper used distance to Wittenberg — the cradle of the Reformation — as exogenous variation in Protestant adoption. We use the same logic in reverse: *closer* to Wittenberg implies *less* Catholic, so kmwittenberg is a positive instrument for cath\_share. Interacted with Post, it instruments cath\_share x Post.

The exclusion restriction is that distance to Wittenberg in 1517 affects 1873–90 fertility outcomes *only through* the Catholic-share channel. Plausible if no other geographic confounder operating with the right timing exists.

In [ ]:
print("=" * 75)
print("2SLS DiD: kmwittenberg x Post as instrument for cath_share x Post")
print("=" * 75)
for outcome in ("cbr", "legitimate_br", "illegitimacy_ratio", "marriage_rate"):
    r = run_iv_did(panel, outcome=outcome, instrument="kmwittenberg")
    iv_star = "***" if r["iv_p"] < .01 else "**" if r["iv_p"] < .05 else "*" if r["iv_p"] < .10 else ""
    ols_star = "***" if r["ols_p"] < .01 else "**" if r["ols_p"] < .05 else "*" if r["ols_p"] < .10 else ""
    print(f"\n  {outcome:>20s}:")
    print(f"      OLS:  {r['ols_coef']:+.5f}{ols_star:<3} (SE={r['ols_se']:.5f})")
    print(f"      2SLS: {r['iv_coef']:+.5f}{iv_star:<3} (SE={r['iv_se']:.5f}), "
          f"first-stage F={r['first_stage_f']:.1f}, partial R²={r['first_stage_partial_r2']:.3f}")

**Interpretation — a major OLS–IV gap.** Across all four outcomes the 2SLS estimates are *much larger in magnitude* than the OLS estimates (CBR: $-0.039^{***}$ vs $-0.000$; marriage: $-0.008^{***}$ vs $-0.004^{***}$). First-stage $F$ is 24.7 — above the rule-of-thumb threshold of 10.

Two interpretations of the OLS–IV divergence:
1. **Attenuation bias.** OLS suffers from measurement error in cath_share or omitted-variable bias toward zero. IV corrects for this and reveals the true effect.
2. **LATE.** The IV identifies a *local* average treatment effect for compliers — counties whose Catholicness is most strongly explained by distance to Wittenberg. This sub-population may be different from the general population.

The Wooldridge over-identification test in section 7 helps discriminate: if the second instrument gives the same answer, story (1) is more credible.

## 7. Multi-instrument IV: Wittenberg + Bishop's seat (Wooldridge over-id)

We add a second instrument: distance to the nearest 1871-Prussian Catholic bishop's seat. Logic is opposite to Wittenberg — *closer* to a bishop's seat implies *more* Catholic institutional infrastructure. With two instruments for one endogenous regressor, we have an over-identifying restriction whose validity can be tested (Wooldridge under clustered SEs, equivalent to Hansen J under homoscedasticity).

Failure to reject the over-id null is consistent with both instruments being valid (= same treatment effect through both channels).

In [ ]:
print("=" * 75)
print("MULTI-INSTRUMENT 2SLS: Wittenberg + Bishop instruments")
print("=" * 75)
for outcome in ("cbr", "legitimate_br", "illegitimacy_ratio", "marriage_rate"):
    r = run_iv_did_multi(panel_with_bishop, outcome=outcome)
    iv_star = "***" if r["iv_p"] < .01 else "**" if r["iv_p"] < .05 else "*" if r["iv_p"] < .10 else ""
    print(f"\n  {outcome:>20s}:")
    print(f"      2SLS: {r['iv_coef']:+.5f}{iv_star:<3} (SE={r['iv_se']:.5f})")
    print(f"      First-stage F: {r['first_stage_f']:.1f}, partial R²: {r['first_stage_partial_r2']:.3f}")
    print(f"      Wooldridge over-id: stat={r['j_stat']:.2f}, df={r['j_df']}, p={r['j_p']:.3f}")

**Interpretation — the cleanest evidence in the notebook.**

| Outcome | Wooldridge over-id $p$ | Verdict |
|---|---|---|
| CBR | $0.000$ | **Rejected** — instruments give different IV coefs. |
| Legitimate BR | $0.000$ | Rejected. |
| Illegitimacy ratio | $\sim 0$ | Rejected. |
| **Marriage rate** | **$0.121$** | **Not rejected** — instruments consistent. |

Marriage rate is the *only* outcome that survives the over-identification test. With both instruments giving consistent estimates, we have multiply-identified evidence that the Kulturkampf depressed marriage rates in Catholic counties. For CBR (and other fertility outcomes), the IV story is internally inconsistent — either the instruments differ in their LATE or one is invalid — so the headline 2SLS estimate cannot be taken at face value.

First-stage $F$ rises to ~176 with two instruments (joint relevance is very strong).

## 8. Conley spatial HAC standard errors

Cluster-robust SEs at the county level allow arbitrary serial correlation within county but assume independence *across* counties. Demographic shocks may have spatial correlation (epidemics, regional grain shocks, common labour-market dynamics). We correct for this using Conley (1999) spatial HAC with a 200 km Bartlett kernel cutoff.

In [ ]:
print("=" * 75)
print("CONLEY HAC SEs (200 km cutoff) vs cluster-robust SEs")
print("=" * 75)
for outcome in ("cbr", "legitimate_br", "illegitimacy_ratio", "marriage_rate"):
    r = spatial_did_se(panel, outcome=outcome, cutoff_km=200)
    cl = r["cluster_se"]["cath_share_x_post"]
    co = r["conley_se"]["cath_share_x_post"]
    print(f"  {outcome:>20s}: beta={r['coef']['cath_share_x_post']:+.5f}, "
          f"cluster SE={cl:.5f}, Conley SE={co:.5f}, ratio={co/cl:.2f}")

**Interpretation.** Conley HAC standard errors come out *smaller* than the cluster-robust ones in three of the four outcomes (illegitimacy ratio drops most, by ~40%). The pattern reflects that, after entity + year FE absorb the bulk of regional variation, the residual idiosyncratic shocks are not strongly spatially correlated; cluster-robust SEs are conservatively wide.

Marriage rate's SE is essentially identical between the two corrections, confirming that the marriage finding is robust to whichever inference correction is used. None of the conclusions change.

## 9. Counterfactual fertility paths (using IV CBR coef)

For each county-year, subtract the IV-attributed Kulturkampf component from the observed CBR. Plot observed and counterfactual ("absent the Kulturkampf") paths separately by Catholic-share group.

In [ ]:
iv_cbr = run_iv_did(panel, outcome="cbr", instrument="kmwittenberg")
fig, ax = plot_counterfactual_paths(
    panel, iv_coef=iv_cbr["iv_coef"], outcome="cbr",
    savepath=str(OUTPUTS / "fig_counterfactual.png"),
)
plt.show()
print(f"\nIV CBR coefficient used: {iv_cbr['iv_coef']:+.5f} (SE = {iv_cbr['iv_se']:.5f})")

**Interpretation.** The dashed lines are the counterfactual: what CBR *would have been* for high- and low-Catholic counties had the Kulturkampf not happened, using the 2SLS estimate to net out its attributable contribution.

Visual takeaway: the high-Catholic dashed line lies *above* the observed solid line in the post-1873 period — i.e. the IV says the Kulturkampf prevented high-Catholic counties from continuing their pre-1873 fertility climb. The low-Catholic counterfactual is essentially indistinguishable from the observed (treatment effect $\approx 0$ for low-Catholic counties).

This figure should be paired with the magnitude decomposition table in notebook 03 to give a complete picture of "what does this 2SLS estimate mean in plain language?".

## 10. End-to-end summary

The four-notebook arc:

1. **Notebook 01.** Build the panel; document the bimodal cath\_share distribution; flag pre-trend warning visually.
2. **Notebook 02.** Establish that marriage rate is the only TWFE-robust outcome. Document non-zero pre-trends; quantify fragility via Honest DiD; verify dCDH weights are benign.
3. **Notebook 03.** The "Catholic effect" is a Polish-province effect. Triple-difference confirms statistically. Jewish-share placebo is *not* null — caveat. Wild bootstrap shows German Catholic counties responded *positively*.
4. **Notebook 04.** Spatial maps + IV identification. Marriage rate passes the Wooldridge over-id test with two instruments — multiply identified. CBR fails. Conley HAC SEs don't change conclusions. The counterfactual figure visualises what the IV says the Kulturkampf "prevented" rather than caused.

**The defensible headline for the paper.** *The Kulturkampf reduced marriage rates in Catholic Prussian counties by approximately 0.4–0.8 per 1,000, robust to TWFE / county-trends / long-difference / 2SLS with two instruments / wild bootstrap / Anderson FDR. The fertility effect is concentrated in the Polish provinces, where it is plausibly attributable to the parallel Germanisation campaign rather than purely religious-institutional disruption. Effects on the German Catholic majority are small or even positive in sign.*